# 1.1 Survival analysis models


In [ ]:
%pip install lifelines

## Data

In [15]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_dir = Path("../data")
horizons = [12, 24, 48, 72]
target_cols = ["time_to_hit_hours", "event"]

train_base_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")
features = [c for c in train_base_df.columns if c not in ["event_id", *target_cols]]

In [16]:
train_df, val_df = train_test_split(
    train_base_df,
    test_size=0.2,
    random_state=42,
    stratify=train_base_df["event"],
)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(train_df[features]), columns=features, index=train_df.index)
X_val = pd.DataFrame(scaler.transform(val_df[features]), columns=features, index=val_df.index)

print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
print("number of features:", len(features))

train shape: (176, 37)
val shape: (45, 37)
number of features: 34


## Models 

### Kaplan-Meier

In [17]:
from lifelines import KaplanMeierFitter

def fit_km(df):
    kmf = KaplanMeierFitter()
    return kmf.fit(df["time_to_hit_hours"], event_observed=df["event"])

def km_event_probs(kmf, df):
    return {h: [1 - kmf.survival_function_at_times(h).iloc[0]] * len(df) for h in horizons}

### Cox Model

In [18]:
from lifelines import CoxPHFitter

def fit_cox(df, penalizer=0.1):
    cph = CoxPHFitter(penalizer=penalizer)
    return cph.fit(df, duration_col="time_to_hit_hours", event_col="event")


def cox_event_probs(cph, df):
    s = cph.predict_survival_function(df[features], times=horizons)
    return {h: 1 - s.loc[h].values for h in horizons}

## Experiments

In [19]:
from lifelines.utils import concordance_index


def label_at_horizon(df, h):
    y = pd.Series(0, index=df.index, dtype="float")
    y[(df["event"] == 1) & (df["time_to_hit_hours"] <= h)] = 1
    y[(df["event"] == 0) & (df["time_to_hit_hours"] < h)] = pd.NA
    return y


def brier(df, preds, h):
    y = label_at_horizon(df, h)
    mask = y.notna()
    return ((y[mask] - pd.Series(preds, index=df.index)[mask]) ** 2).mean()


def evaluate(df, preds):
    cidx = concordance_index(
        df["time_to_hit_hours"], -pd.Series(preds[72]), df["event"]
    )
    wb = (
        0.3 * brier(df, preds[24], 24)
        + 0.4 * brier(df, preds[48], 48)
        + 0.3 * brier(df, preds[72], 72)
    )
    return {
        "c_index": cidx,
        "weighted_brier": wb,
        "hybrid": 0.3 * cidx + 0.7 * (1 - wb),
    }

In [20]:
kmf = fit_km(train_df)
km_preds = km_event_probs(kmf, val_df)

evaluate(val_df, km_preds)

{'c_index': np.float64(0.5),
 'weighted_brier': np.float64(0.22828619440302028),
 'hybrid': np.float64(0.6901996639178858)}

In [21]:
cox_train_df = pd.concat([train_df[target_cols], X_train], axis=1)
cox_val_df = pd.concat([val_df[target_cols], X_val], axis=1)

cph = fit_cox(cox_train_df, penalizer=0.1)
cox_preds = cox_event_probs(cph, cox_val_df)

evaluate(cox_val_df, cox_preds)

{'c_index': np.float64(0.8817427385892116),
 'weighted_brier': np.float64(0.10390377590435258),
 'hybrid': np.float64(0.8917901784437166)}